# StressSystems — worked examples

This notebook runs on an **R** runtime. In Colab choose *Runtime → Change runtime type → R*.

Contents:
1. Installation
2. Defining distributions (built-in, by name, user-defined)
3. Reliability R = P(cY < X)
4. Multicomponent systems
5. Parameter estimation
6. Estimating R with confidence intervals
7. Monte Carlo studies

## 1. Installation

In [ ]:
if (!requireNamespace("remotes", quietly = TRUE)) install.packages("remotes")
remotes::install_github("yamenetoo/Statistical-Repository", subdir = "StressSystems")
library(StressSystems)

## 2. Defining distributions

Two families are built in: Topp–Leone (`toppleone`, parameter `shape`) and the generalized Rayleigh / Burr type X (`genrayleigh`, parameters `shape` and `rate`).

In [ ]:
X <- ss_dist("toppleone")
Y <- ss_dist("genrayleigh")
X
Y

Any family following R's `d/p/q/r` convention can be used by name. The parameter names are read from the density.

In [ ]:
W <- ss_dist("weibull", par = c(shape = 2, scale = 1.5))
G <- ss_dist("gamma")            # parameters: shape, rate
N <- ss_dist("norm")             # mean is real, sd is positive
W; G; N

A user-defined family only needs a density and a CDF. Here is the Lindley distribution with
$f(x)=\frac{\theta^2}{1+\theta}(1+x)e^{-\theta x}$ and $F(x)=1-\frac{1+\theta+\theta x}{1+\theta}e^{-\theta x}$.
Quantiles and random numbers are then obtained by numerical inversion.

In [ ]:
dlindley <- function(x, theta, log = FALSE) {
  d <- ifelse(x > 0, theta^2 / (1 + theta) * (1 + x) * exp(-theta * x), 0)
  if (log) log(d) else d
}
plindley <- function(q, theta) {
  ifelse(q > 0, 1 - (1 + theta + theta * q) / (1 + theta) * exp(-theta * q), 0)
}
L <- ss_dist(d = dlindley, p = plindley, lower = 0, upper = Inf,
             constraints = list(theta = "positive"))
L

Built-in densities and random generation:

In [ ]:
curve(dtoppleone(x, 0.5), 0.001, 0.999, ylab = "density", main = "Topp-Leone")
curve(dtoppleone(x, 2), add = TRUE, lty = 2)
legend("top", c("shape = 0.5", "shape = 2"), lty = 1:2)
hist(rgenrayleigh(2000, shape = 1.5, rate = 2), breaks = 40, freq = FALSE,
     main = "Generalized Rayleigh", xlab = "x")
curve(dgenrayleigh(x, 1.5, 2), add = TRUE, lwd = 2)

## 3. Reliability R = P(cY < X)

In [ ]:
# Check against a closed form: X ~ Exp(2), Y ~ Exp(3)  =>  R = 3/5
Xe <- ss_dist("exp", par = c(rate = 2))
Ye <- ss_dist("exp", par = c(rate = 3))
ss_R(Xe, Ye)

In [ ]:
# Topp-Leone strength, generalized Rayleigh stress
ss_R(X, Y, par_x = c(shape = 0.7), par_y = c(shape = 1, rate = 2))

# Stress amplified (c > 1) or attenuated (c < 1)
sapply(c(0.5, 1, 2), function(cc)
  ss_R(X, Y, par_x = c(shape = 0.7), par_y = c(shape = 1, rate = 2), c = cc))

In [ ]:
# Mixing families: Weibull strength vs Lindley stress
ss_R(W, L, par_y = c(theta = 2))

In [ ]:
# R as a function of the strength parameter
v <- seq(0.1, 3, length.out = 40)
Rv <- sapply(v, function(s) ss_R(X, Y, c(shape = s), c(shape = 1, rate = 2)))
plot(v, Rv, type = "l", lwd = 2, xlab = "Topp-Leone shape", ylab = "R",
     main = "R = P(Y < X)")

## 4. Multicomponent systems

In [ ]:
px <- c(shape = 0.7)
py <- c(shape = 1, rate = 2)

# One component under two stresses (shared component)
stresses <- data.frame(shape = c(1, 2), rate = c(2, 3))
ss_multistress(X, Y, par_x = px, par_y = stresses)

# Each stress on its own independent component
ss_multistress(X, Y, par_x = px, par_y = stresses, components = "independent")

In [ ]:
# P(Z < Y < X): stress between a lower and an upper strength
ss_R3(lower = X, stress = Y, upper = X,
      par_z = c(shape = 0.3), par_y = py, par_x = c(shape = 0.9))

In [ ]:
# s-out-of-k system: at least s of k components must survive
sapply(1:3, function(s) ss_sk(X, Y, s = s, k = 3, par_x = px, par_y = py))

In [ ]:
# Cascade system with attenuation factor c
ss_cascade(X, Y, n = 3, c = 0.8, par_x = px, par_y = py)

## 5. Parameter estimation

In [ ]:
set.seed(1)
x <- rtoppleone(30, shape = 0.7)
y <- rgenrayleigh(30, shape = 1, rate = 2)

methods <- c("mle", "mps", "lse", "wlse", "cvm", "ad")
t(sapply(methods, function(m) ss_fit(y, Y, method = m)$estimate))

In [ ]:
ss_fit(x, X, method = "mle")
ss_fit(x, X, method = "jackknife")
ss_fit(x, X, method = "bootstrap", B = 200)

In [ ]:
# Bayesian estimation with Gamma(1, 1) priors
fb <- ss_bayes(y, Y, prior = function(th) sum(dgamma(th, 1, 1, log = TRUE)),
               iter = 5000, burnin = 1000)
fb
par(mfrow = c(1, 2))
plot(fb$draws[, "shape"], type = "l", main = "shape", ylab = "")
plot(fb$draws[, "rate"], type = "l", main = "rate", ylab = "")
par(mfrow = c(1, 1))

## 6. Estimating R with confidence intervals

In [ ]:
true_R <- ss_R(X, Y, c(shape = 0.7), c(shape = 1, rate = 2))
true_R

ss_estimate_R(x, y, X, Y, method = "mle")                       # delta method
ss_estimate_R(x, y, X, Y, method = "mle", ci = "boot", B = 300) # bootstrap
ss_estimate_R(x, y, X, Y, method = "bayes", iter = 4000, burnin = 1000)

## 7. Monte Carlo studies

`MC` is kept small here so the notebook runs quickly; use `MC = 1000` for a paper.

In [ ]:
sim <- ss_simulate(X, Y, par_x = c(shape = 0.7), par_y = c(shape = 1, rate = 2),
                   n = c(10, 25, 50), MC = 50,
                   methods = c("mle", "mps", "jackknife"), seed = 2026)
sim
head(sim$parameters)

In [ ]:
s <- sim$summary
for (m in unique(s$method)) {
  d <- s[s$method == m, ]
  if (m == unique(s$method)[1])
    plot(d$n_x, d$mse, type = "b", log = "y", ylim = range(s$mse),
         xlab = "n", ylab = "MSE of R", main = "MSE of the estimators of R")
  else lines(d$n_x, d$mse, type = "b", lty = match(m, unique(s$method)))
}
legend("topright", unique(s$method), lty = seq_along(unique(s$method)))

In [ ]:
# Grid of scenarios (small version of the full study)
grid <- ss_simulate_grid(X, Y,
                         grid_x = data.frame(shape = c(0.3, 0.9)),
                         grid_y = data.frame(shape = c(0.5, 2), rate = c(2, 0.5)),
                         n = c(10, 25), MC = 30, methods = c("mle", "mps"), seed = 1)
grid$summary